# Análise da execução financeira por função

Depois de entender a estrutura da base no notebook de diagnóstico, agora vou começar a olhar para o comportamento das despesas. O objetivo aqui é comparar o que foi empenhado com o que foi pago, sempre com cuidado para não misturar funções com totais ou subfunções.

Neste notebook, vou trabalhar principalmente com linhas em que `tipo_conta == "função"`. Isso evita dupla contagem e deixa a comparação mais direta: cada linha representa uma grande área de atuação do governo, como Saúde, Educação, Urbanismo ou Administração.

Também vou tratar 2025 como ano parcial. Ele aparece na base, mas não deve entrar nas comparações históricas principais junto com os anos completos.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def formatar_numero_br(valor):
    return f"{valor:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")


pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", formatar_numero_br)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dados_processados").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_PATH = PROJECT_ROOT / "dados_processados" / "finbra_consolidado.parquet"
ANOS_COMPLETOS = [2020, 2021, 2022, 2023, 2024]
ANO_REFERENCIA = 2024

BASE_PATH.relative_to(PROJECT_ROOT)

PosixPath('dados_processados/finbra_consolidado.parquet')

In [2]:
df = pd.read_parquet(BASE_PATH)

df["capital"] = df["Instituição"].str.replace(
    r"^Prefeitura Municipal (?:de |do |da )", "", regex=True
)

df.head()

,ano,Instituição,Cod.IBGE,UF,População,Coluna,Conta,tipo_conta,codigo_conta,nome_conta,codigo_funcao,nome_funcao,codigo_subfuncao,nome_subfuncao,Identificador da Conta,Valor,capital
0,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,Despesas Exceto Intraorçamentárias,total,<NA>,Despesas Exceto Intraorçamentárias,<NA>,<NA>,<NA>,<NA>,siconfi-cor_TotalDespesas,"874.885.274,98",Rio Branco - AC
1,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,função,01,Legislativa,01,Legislativa,<NA>,<NA>,siconfi-cor_TotalDespesas,"29.014.059,54",Rio Branco - AC
2,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54",Rio Branco - AC
3,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,função,03,Essencial à Justiça,03,Essencial à Justiça,<NA>,<NA>,siconfi-cor_TotalDespesas,"18.822.895,07",Rio Branco - AC
4,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03.092 - Representação Judicial e Extrajudicial,subfunção,03.092,Representação Judicial e Extrajudicial,03,Essencial à Justiça,092,Representação Judicial e Extrajudicial,siconfi-cor_TotalDespesas,"18.822.895,07",Rio Branco - AC


## Preparando a base de execução por função

Para calcular a taxa de execução, preciso colocar `Despesas Empenhadas` e `Despesas Pagas` lado a lado. Vou filtrar apenas as linhas de função e transformar os estágios da despesa em colunas. Assim cada registro passa a representar uma combinação de ano, capital e função.

In [3]:
estagios_interesse = ["Despesas Empenhadas", "Despesas Pagas"]

base_funcoes = df[
    (df["tipo_conta"] == "função")
    & (df["Coluna"].isin(estagios_interesse))
].copy()

indice_execucao = [
    "ano",
    "Instituição",
    "capital",
    "Cod.IBGE",
    "UF",
    "População",
    "codigo_funcao",
    "nome_funcao",
]

execucao_funcao = (
    base_funcoes.pivot_table(
        index=indice_execucao,
        columns="Coluna",
        values="Valor",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename(
        columns={
            "Despesas Empenhadas": "valor_empenhado",
            "Despesas Pagas": "valor_pago",
        }
    )
)

for coluna in ["valor_empenhado", "valor_pago"]:
    if coluna not in execucao_funcao.columns:
        execucao_funcao[coluna] = 0.0

execucao_funcao["taxa_execucao"] = np.where(
    execucao_funcao["valor_empenhado"] > 0,
    execucao_funcao["valor_pago"] / execucao_funcao["valor_empenhado"],
    np.nan,
)
execucao_funcao["taxa_execucao_%"] = execucao_funcao["taxa_execucao"] * 100
execucao_funcao["diferenca_empenhado_pago"] = (
    execucao_funcao["valor_empenhado"] - execucao_funcao["valor_pago"]
)
execucao_funcao["valor_pago_per_capita"] = (
    execucao_funcao["valor_pago"] / execucao_funcao["População"]
)
execucao_funcao["ano_parcial"] = ~execucao_funcao["ano"].isin(ANOS_COMPLETOS)

execucao_funcao.head()

Coluna,ano,Instituição,capital,Cod.IBGE,UF,População,codigo_funcao,nome_funcao,valor_empenhado,valor_pago,taxa_execucao,taxa_execucao_%,diferenca_empenhado_pago,valor_pago_per_capita,ano_parcial
0,2020,Prefeitura Municipal de Aracaju - SE,Aracaju - SE,2800308,SE,657013,01,Legislativa,"55.657.017,67","53.161.537,18","0,96","95,52","2.495.480,49","80,91",False
1,2020,Prefeitura Municipal de Aracaju - SE,Aracaju - SE,2800308,SE,657013,04,Administração,"270.109.344,07","266.689.907,93","0,99","98,73","3.419.436,14","405,91",False
2,2020,Prefeitura Municipal de Aracaju - SE,Aracaju - SE,2800308,SE,657013,06,Segurança Pública,"2.113.472,07","1.975.443,99","0,93","93,47","138.028,08","3,01",False
3,2020,Prefeitura Municipal de Aracaju - SE,Aracaju - SE,2800308,SE,657013,08,Assistência Social,"56.536.370,46","52.129.704,33","0,92","92,21","4.406.666,13","79,34",False
4,2020,Prefeitura Municipal de Aracaju - SE,Aracaju - SE,2800308,SE,657013,09,Previdência Social,"317.858.492,45","317.686.107,57","1,00","99,95","172.384,88","483,53",False


Com essa tabela, consigo responder a pergunta central do desafio: de tudo que foi empenhado em uma função, quanto foi pago dentro do mesmo ano? Antes de ir para Saúde e Educação, vou olhar alguns rankings gerais para entender a variação entre capitais e funções.

In [4]:
resumo_execucao = pd.DataFrame(
    {
        "métrica": [
            "linhas de execução por função",
            "anos completos usados na análise principal",
            "capitais em 2024",
            "funções em 2024",
        ],
        "valor": [
            len(execucao_funcao),
            len(ANOS_COMPLETOS),
            execucao_funcao.loc[execucao_funcao["ano"] == ANO_REFERENCIA, "Cod.IBGE"].nunique(),
            execucao_funcao.loc[execucao_funcao["ano"] == ANO_REFERENCIA, "codigo_funcao"].nunique(),
        ],
    }
)

resumo_execucao

,métrica,valor
0,linhas de execução por função,2806
1,anos completos usados na análise principal,5
2,capitais em 2024,26
3,funções em 2024,27


## Ranking das capitais em 2024

Vou usar 2024 como ano de referência porque é o ano completo mais recente. Aqui, somo as funções de cada capital e calculo a taxa geral de execução financeira. Essa leitura não explica sozinha o motivo das diferenças, mas ajuda a enxergar quem pagou uma proporção maior ou menor do que empenhou.

In [5]:
ranking_capitais_2024 = (
    execucao_funcao[execucao_funcao["ano"] == ANO_REFERENCIA]
    .groupby(["capital", "UF", "Cod.IBGE"], as_index=False)
    .agg(
        populacao=("População", "first"),
        valor_empenhado=("valor_empenhado", "sum"),
        valor_pago=("valor_pago", "sum"),
    )
)

ranking_capitais_2024["taxa_execucao"] = (
    ranking_capitais_2024["valor_pago"] / ranking_capitais_2024["valor_empenhado"]
)
ranking_capitais_2024["taxa_execucao_%"] = ranking_capitais_2024["taxa_execucao"] * 100
ranking_capitais_2024["valor_pago_per_capita"] = (
    ranking_capitais_2024["valor_pago"] / ranking_capitais_2024["populacao"]
)

ranking_capitais_2024.sort_values("taxa_execucao", ascending=False).head(10)

,capital,UF,Cod.IBGE,populacao,valor_empenhado,valor_pago,taxa_execucao,taxa_execucao_%,valor_pago_per_capita
2,Belém - PA,PA,1501402,1367336,"5.577.035.959,33","5.485.862.857,97","0,98","98,37","4.012,08"
13,Manaus - AM,AM,1302603,2054731,"10.303.853.473,17","10.114.580.004,98","0,98","98,16","4.922,58"
0,Aracaju - SE,SE,2800308,605309,"3.292.587.894,53","3.221.705.809,53","0,98","97,85","5.322,42"
15,Palmas - TO,TO,1721000,334454,"2.149.999.077,21","2.100.444.945,18","0,98","97,70","6.280,22"
18,Recife - PE,PE,2611606,1494586,"8.973.382.427,47","8.762.894.499,47","0,98","97,65","5.863,09"
9,Goiânia - GO,GO,5208707,1414483,"9.084.332.557,17","8.858.800.770,66","0,98","97,52","6.262,92"
8,Fortaleza - CE,CE,2304400,2596157,"13.389.864.271,76","13.039.432.082,46","0,97","97,38","5.022,59"
21,Salvador - BA,BA,2927408,2610987,"12.250.033.162,38","11.916.245.750,70","0,97","97,28","4.563,89"
19,Rio Branco - AC,AC,1200401,364368,"1.928.793.136,47","1.861.094.968,22","0,96","96,49","5.107,73"
11,Macapá - AP,AP,1600303,478448,"2.149.044.128,45","2.061.414.591,93","0,96","95,92","4.308,54"


In [6]:
ranking_capitais_2024.sort_values("taxa_execucao", ascending=True).head(10)

,capital,UF,Cod.IBGE,populacao,valor_empenhado,valor_pago,taxa_execucao,taxa_execucao_%,valor_pago_per_capita
14,Natal - RN,RN,2408102,751932,"4.188.079.695,21","3.618.430.249,50","0,86","86,40","4.812,18"
6,Curitiba - PR,PR,4106902,1871789,"12.593.744.597,37","11.072.650.007,83","0,88","87,92","5.915,54"
22,São Luís - MA,MA,2111300,1061374,"5.291.320.191,76","4.657.244.817,53","0,88","88,02","4.387,94"
25,Vitória - ES,ES,3205309,331785,"3.291.918.600,79","2.913.280.844,74","0,88","88,50","8.780,63"
5,Cuiabá - MT,MT,5103403,694244,"3.883.105.849,87","3.493.769.014,18","0,90","89,97","5.032,48"
1,Belo Horizonte - MG,MG,3106200,2392678,"18.189.047.925,18","16.475.284.385,62","0,91","90,58","6.885,71"
16,Porto Alegre - RS,RS,4314902,1404269,"9.894.116.859,49","8.974.553.709,16","0,91","90,71","6.390,91"
4,Campo Grande - MS,MS,5002704,942140,"5.879.022.739,20","5.343.562.196,08","0,91","90,89","5.671,73"
12,Maceió - AL,AL,2704302,960667,"5.085.158.644,86","4.699.868.969,77","0,92","92,42","4.892,30"
7,Florianópolis - SC,SC,4205407,574200,"3.625.153.996,53","3.373.052.982,80","0,93","93,05","5.874,35"


## Ranking das funções em 2024

Agora vou mudar o ponto de vista: em vez de olhar capital por capital, quero ver quais funções tiveram maior ou menor taxa de execução no conjunto das capitais. Isso ajuda a encontrar áreas em que a diferença entre empenhado e pago parece mais forte.

In [7]:
ranking_funcoes_2024 = (
    execucao_funcao[execucao_funcao["ano"] == ANO_REFERENCIA]
    .groupby(["codigo_funcao", "nome_funcao"], as_index=False)
    .agg(
        capitais=("Cod.IBGE", "nunique"),
        populacao_base=("População", "sum"),
        valor_empenhado=("valor_empenhado", "sum"),
        valor_pago=("valor_pago", "sum"),
    )
)

ranking_funcoes_2024["taxa_execucao"] = (
    ranking_funcoes_2024["valor_pago"] / ranking_funcoes_2024["valor_empenhado"]
)
ranking_funcoes_2024["taxa_execucao_%"] = ranking_funcoes_2024["taxa_execucao"] * 100
ranking_funcoes_2024["valor_pago_per_capita"] = (
    ranking_funcoes_2024["valor_pago"] / ranking_funcoes_2024["populacao_base"]
)

ranking_funcoes_2024.sort_values("taxa_execucao", ascending=True).head(10)

,codigo_funcao,nome_funcao,capitais,populacao_base,valor_empenhado,valor_pago,taxa_execucao,taxa_execucao_%,valor_pago_per_capita
19,20,Agricultura,15,27165919,"215.895.243,76","180.621.010,04","0,84","83,66","6,65"
16,17,Saneamento,24,44065940,"9.354.122.624,37","7.975.499.790,75","0,85","85,26","180,99"
14,15,Urbanismo,26,45760012,"35.992.554.221,97","31.590.952.544,98","0,88","87,77","690,36"
21,23,Comércio e Serviços,26,45760012,"2.180.188.850,75","1.931.009.235,79","0,89","88,57","42,20"
17,18,Gestão Ambiental,26,45760012,"2.622.224.230,19","2.339.595.079,59","0,89","89,22","51,13"
12,13,Cultura,26,45760012,"3.006.753.573,32","2.705.235.184,86","0,90","89,97","59,12"
15,16,Habitação,24,44220190,"6.466.572.913,70","5.825.024.933,35","0,90","90,08","131,73"
4,05,Defesa Nacional,3,14733359,"2.429.774,74","2.198.262,98","0,90","90,47","0,15"
18,19,Ciência e Tecnologia,16,36662160,"455.042.087,78","414.576.549,53","0,91","91,11","11,31"
22,24,Comunicações,9,22334031,"529.162.685,18","489.088.616,59","0,92","92,43","21,90"


Esse ranking por função é um primeiro mapa de onde investigar. Funções com taxa menor podem indicar mais valores ficando para restos a pagar, mas também podem refletir características próprias de obras, contratos, sazonalidade ou forma de registro. Por isso, nesta etapa eu trato o ranking como ponto de partida, não como conclusão final.

## Saúde e Educação entre 2020 e 2024

Agora vou focar em Saúde e Educação, duas funções centrais para a leitura do orçamento municipal. Para evitar a distorção de 2025 incompleto, vou usar apenas os anos completos de 2020 a 2024.

In [8]:
funcoes_prioritarias = ["Saúde", "Educação"]

saude_educacao = execucao_funcao[
    execucao_funcao["ano"].isin(ANOS_COMPLETOS)
    & execucao_funcao["nome_funcao"].isin(funcoes_prioritarias)
].copy()

evolucao_saude_educacao = (
    saude_educacao.groupby(["ano", "nome_funcao"], as_index=False)
    .agg(
        capitais=("Cod.IBGE", "nunique"),
        populacao_base=("População", "sum"),
        valor_empenhado=("valor_empenhado", "sum"),
        valor_pago=("valor_pago", "sum"),
    )
)

evolucao_saude_educacao["taxa_execucao"] = (
    evolucao_saude_educacao["valor_pago"] / evolucao_saude_educacao["valor_empenhado"]
)
evolucao_saude_educacao["taxa_execucao_%"] = evolucao_saude_educacao["taxa_execucao"] * 100
evolucao_saude_educacao["valor_pago_per_capita"] = (
    evolucao_saude_educacao["valor_pago"] / evolucao_saude_educacao["populacao_base"]
)

evolucao_saude_educacao

,ano,nome_funcao,capitais,populacao_base,valor_empenhado,valor_pago,taxa_execucao,taxa_execucao_%,valor_pago_per_capita
0,2020,Educação,26,47124865,"35.581.988.578,38","31.153.808.990,10","0,88","87,55","661,09"
1,2020,Saúde,26,47124865,"46.226.962.461,66","42.563.764.394,09","0,92","92,08","903,21"
2,2021,Educação,26,47479406,"43.068.136.888,04","35.200.352.028,98","0,82","81,73","741,38"
3,2021,Saúde,26,47479406,"50.131.935.354,20","46.397.972.632,28","0,93","92,55","977,22"
4,2022,Educação,26,47821713,"51.326.666.136,42","43.213.885.633,88","0,84","84,19","903,65"
5,2022,Saúde,26,47821713,"55.204.786.157,49","51.628.271.411,45","0,94","93,52","1.079,60"
6,2023,Educação,26,47821713,"54.394.069.396,31","49.371.360.754,10","0,91","90,77","1.032,40"
7,2023,Saúde,26,47821713,"62.588.857.392,46","58.564.697.943,62","0,94","93,57","1.224,65"
8,2024,Educação,26,45760012,"61.711.408.856,02","57.935.658.616,65","0,94","93,88","1.266,08"
9,2024,Saúde,26,45760012,"70.504.158.203,39","66.558.535.743,87","0,94","94,40","1.454,51"


A tabela acima mostra o comportamento agregado das capitais. O próximo passo é trazer Maceió para essa mesma régua, comparando a capital com a média e a mediana das demais capitais em cada ano.

In [9]:
maceio = saude_educacao[
    saude_educacao["capital"].str.contains("Maceió", case=False, na=False)
].copy()

demais_capitais = saude_educacao[
    ~saude_educacao["capital"].str.contains("Maceió", case=False, na=False)
].copy()

referencia_capitais = (
    demais_capitais.groupby(["ano", "nome_funcao"], as_index=False)
    .agg(
        taxa_mediana_capitais=("taxa_execucao", "median"),
        taxa_media_capitais=("taxa_execucao", "mean"),
        pago_pc_mediana_capitais=("valor_pago_per_capita", "median"),
        pago_pc_media_capitais=("valor_pago_per_capita", "mean"),
    )
)

comparacao_maceio = maceio.merge(
    referencia_capitais,
    on=["ano", "nome_funcao"],
    how="left",
)

comparacao_maceio["taxa_execucao_%"] = comparacao_maceio["taxa_execucao"] * 100
comparacao_maceio["taxa_mediana_capitais_%"] = (
    comparacao_maceio["taxa_mediana_capitais"] * 100
)
comparacao_maceio["dif_taxa_vs_mediana_pontos"] = (
    comparacao_maceio["taxa_execucao"] - comparacao_maceio["taxa_mediana_capitais"]
) * 100
comparacao_maceio["dif_pago_pc_vs_mediana"] = (
    comparacao_maceio["valor_pago_per_capita"]
    - comparacao_maceio["pago_pc_mediana_capitais"]
)

comparacao_maceio[
    [
        "ano",
        "nome_funcao",
        "valor_empenhado",
        "valor_pago",
        "taxa_execucao_%",
        "taxa_mediana_capitais_%",
        "dif_taxa_vs_mediana_pontos",
        "valor_pago_per_capita",
        "pago_pc_mediana_capitais",
        "dif_pago_pc_vs_mediana",
    ]
].sort_values(["nome_funcao", "ano"])

,ano,nome_funcao,valor_empenhado,valor_pago,taxa_execucao_%,taxa_mediana_capitais_%,dif_taxa_vs_mediana_pontos,valor_pago_per_capita,pago_pc_mediana_capitais,dif_pago_pc_vs_mediana
1,2020,Educação,"357.868.315,31","319.093.839,56","89,17","95,78","-6,61","313,16","606,15","-292,99"
3,2021,Educação,"464.266.926,96","384.922.142,66","82,91","85,32","-2,41","375,40","615,01","-239,61"
5,2022,Educação,"768.448.799,82","724.440.479,18","94,27","89,14","5,14","702,25","796,21","-93,96"
7,2023,Educação,"648.604.839,79","612.559.556,10","94,44","91,53","2,91","593,80","943,21","-349,42"
9,2024,Educação,"803.988.514,26","687.560.500,43","85,52","94,57","-9,05","715,71","1.093,49","-377,78"
0,2020,Saúde,"792.065.782,48","781.472.702,46","98,66","93,13","5,53","766,94","794,60","-27,66"
2,2021,Saúde,"774.134.174,08","755.985.079,24","97,66","94,49","3,17","737,29","879,60","-142,31"
4,2022,Saúde,"872.967.088,35","858.715.266,40","98,37","92,51","5,86","832,41","902,39","-69,98"
6,2023,Saúde,"1.302.781.725,57","1.274.498.167,54","97,83","93,21","4,62","1.235,46","1.066,93","168,53"
8,2024,Saúde,"1.297.135.226,50","1.262.955.548,42","97,36","95,44","1,92","1.314,67","1.278,46","36,20"


## Posição de Maceió em 2024

Para fechar este recorte, vou olhar a posição de Maceió no ano completo mais recente. Aqui uso dois rankings: taxa de execução e valor pago per capita. O primeiro mostra a proporção do empenhado que foi paga; o segundo ajuda a comparar o volume pago considerando o tamanho da população.

In [10]:
ranking_saude_educacao_2024 = saude_educacao[
    saude_educacao["ano"] == ANO_REFERENCIA
].copy()

ranking_saude_educacao_2024["rank_taxa_execucao"] = (
    ranking_saude_educacao_2024.groupby("nome_funcao")["taxa_execucao"]
    .rank(ascending=False, method="min")
    .astype(int)
)
ranking_saude_educacao_2024["rank_pago_per_capita"] = (
    ranking_saude_educacao_2024.groupby("nome_funcao")["valor_pago_per_capita"]
    .rank(ascending=False, method="min")
    .astype(int)
)
ranking_saude_educacao_2024["total_capitais"] = (
    ranking_saude_educacao_2024.groupby("nome_funcao")["Cod.IBGE"].transform("nunique")
)
ranking_saude_educacao_2024["taxa_execucao_%"] = (
    ranking_saude_educacao_2024["taxa_execucao"] * 100
)

posicao_maceio_2024 = ranking_saude_educacao_2024[
    ranking_saude_educacao_2024["capital"].str.contains("Maceió", case=False, na=False)
][
    [
        "ano",
        "capital",
        "nome_funcao",
        "taxa_execucao_%",
        "rank_taxa_execucao",
        "valor_pago_per_capita",
        "rank_pago_per_capita",
        "total_capitais",
    ]
].sort_values("nome_funcao")

posicao_maceio_2024

Coluna,ano,capital,nome_funcao,taxa_execucao_%,rank_taxa_execucao,valor_pago_per_capita,rank_pago_per_capita,total_capitais
2305,2024,Maceió - AL,Educação,"85,52",24,"715,71",25,26
2303,2024,Maceió - AL,Saúde,"97,36",5,"1.314,67",13,26


## Recorte rápido por subfunção em Maceió

Antes de fechar a análise, vou abrir um pouco Saúde e Educação em Maceió para ver quais subfunções concentram o valor pago em 2024. A ideia aqui não é trocar o foco do desafio, que está em função, mas mostrar que a base já permite descer um nível quando isso ajuda a explicar melhor o resultado.

Também vou manter `demais_subfunções` na tabela. Essa linha não é uma subfunção específica; ela funciona como um agrupamento residual das subfunções menores daquela função.

In [11]:
pagas_maceio_2024 = df[
    (df["ano"] == ANO_REFERENCIA)
    & df["capital"].str.contains("Maceió", case=False, na=False)
    & df["nome_funcao"].isin(funcoes_prioritarias)
    & df["tipo_conta"].isin(["subfunção", "demais_subfunções"])
    & (df["Coluna"] == "Despesas Pagas")
].copy()

totais_funcoes_maceio_2024 = execucao_funcao[
    (execucao_funcao["ano"] == ANO_REFERENCIA)
    & execucao_funcao["capital"].str.contains("Maceió", case=False, na=False)
    & execucao_funcao["nome_funcao"].isin(funcoes_prioritarias)
][["codigo_funcao", "nome_funcao", "valor_pago"]].rename(
    columns={"valor_pago": "valor_pago_funcao"}
)

subfuncoes_maceio_2024 = (
    pagas_maceio_2024.groupby(
        [
            "codigo_funcao",
            "nome_funcao",
            "tipo_conta",
            "codigo_conta",
            "nome_conta",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(valor_pago=("Valor", "sum"))
    .merge(totais_funcoes_maceio_2024, on=["codigo_funcao", "nome_funcao"], how="left")
)

subfuncoes_maceio_2024["participacao_no_pago_funcao_%"] = (
    subfuncoes_maceio_2024["valor_pago"]
    / subfuncoes_maceio_2024["valor_pago_funcao"]
    * 100
)

subfuncoes_maceio_2024 = subfuncoes_maceio_2024.sort_values(
    ["nome_funcao", "valor_pago"], ascending=[True, False]
).reset_index(drop=True)

subfuncoes_maceio_2024

,codigo_funcao,nome_funcao,tipo_conta,codigo_conta,nome_conta,valor_pago,valor_pago_funcao,participacao_no_pago_funcao_%
0,12,Educação,subfunção,12.361,Ensino Fundamental,"291.273.033,05","687.560.500,43","42,36"
1,12,Educação,subfunção,12.122,Administração Geral,"193.550.273,99","687.560.500,43","28,15"
2,12,Educação,subfunção,12.368,Educação Básica,"141.609.400,11","687.560.500,43","20,60"
3,12,Educação,subfunção,12.365,Educação Infantil,"50.237.143,03","687.560.500,43","7,31"
4,12,Educação,demais_subfunções,FU12,Demais Subfunções,"6.210.397,33","687.560.500,43","0,90"
5,12,Educação,subfunção,12.367,Educação Especial,"3.440.711,17","687.560.500,43","0,50"
6,12,Educação,subfunção,12.366,Educação de Jovens e Adultos,"1.239.541,75","687.560.500,43","0,18"
7,10,Saúde,subfunção,10.302,Assistência Hospitalar e Ambulatorial,"739.098.739,88","1.262.955.548,42","58,52"
8,10,Saúde,subfunção,10.301,Atenção Básica,"346.872.424,92","1.262.955.548,42","27,47"
9,10,Saúde,subfunção,10.122,Administração Geral,"77.189.022,44","1.262.955.548,42","6,11"


## Arquivos finais da análise

Agora que os recortes principais já estão definidos, vou salvar tabelas e gráficos em `outputs/`. Esses arquivos são úteis para a documentação final porque deixam os principais resultados acessíveis sem depender de abrir o notebook.

In [12]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.gerar_outputs import gerar_outputs

arquivos_gerados = gerar_outputs(BASE_PATH, PROJECT_ROOT / "outputs")

pd.DataFrame(
    {"arquivo": str(caminho.relative_to(PROJECT_ROOT))}
    for caminho in arquivos_gerados.values()
)

,arquivo
0,outputs/tabelas/ranking_capitais_2024.csv
1,outputs/tabelas/ranking_funcoes_2024.csv
2,outputs/tabelas/maceio_saude_educacao_2020_202...
3,outputs/tabelas/maceio_subfuncoes_saude_educac...
4,outputs/figuras/ranking_capitais_taxa_execucao...
5,outputs/figuras/maceio_vs_mediana_taxa_execuca...
6,outputs/figuras/posicao_maceio_saude_educacao_...


## O que levo desta etapa

Agora esta etapa já fecha a análise principal do desafio, em vez de ficar só como exploração.

- A taxa de execução foi calculada usando apenas `tipo_conta == "função"`, o que evita misturar funções com totais, subfunções e linhas agregadas.
- Usei 2024 como ano de referência por ser o ano completo mais recente. O ano de 2025 continua fora das comparações principais porque ainda está parcial.
- No ranking geral de 2024, Maceió aparece abaixo da mediana das capitais na taxa de execução total por função.
- Em Saúde, Maceió tem uma posição forte na taxa de execução em 2024: pagou 97,36% do que empenhou e ficou em 5º lugar entre as 26 capitais.
- Em Educação, o resultado é mais fraco: a taxa de execução ficou em 85,52%, colocando Maceió na 24ª posição entre as capitais.
- No valor pago per capita, Maceió fica em uma posição intermediária em Saúde e entre as últimas em Educação, o que ajuda a mostrar que taxa de execução e volume por habitante contam histórias diferentes.
- O recorte por subfunções mostra onde o gasto pago se concentra dentro de Saúde e Educação, mas esse ponto entra como apoio à interpretação, não como o eixo central do projeto.
- As tabelas e figuras finais foram salvas em `outputs/`, para facilitar a leitura do projeto fora do notebook.